In [ ]:
# TavilySearch 사용
# VectorDB 없이 웹 검색 결과를 문맥으로 넣어 LLM이 답하도록 하는 웹 검색기반 RAG 구조에 적합
# 실시간 웹 검색 구현
!pip install -U langchain-openai langchain-community langchain-tavily

In [ ]:
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch
from langchain_core.prompts import ChatPromptTemplate
import time
import os
from dotenv import load_dotenv

load_dotenv()

In [ ]:
class MyOptimizedWebRag:
    def __init__(self):
        self.search = TavilySearch(
            max_results = 5,
            api_key = os.getenv('TAVILY_API_KEY')
        )

        self.llm = ChatOpenAI(
            model = "gpt-4o-mini",
            temperature=0.0,
            api_key=os.getenv("OPENAI_API_KEY")
        )

        self.summary_prompt = ChatPromptTemplate.from_template(
            """
            당신은 검색결과를 정리 요약하는 전문가입니다.

            다음은 웹 검색 결과입니다.
            {search_results}

            요구사항:
             - 광고/홍보성 내용을 제거하고
             - 서로 겹치는 내용은 합쳐서
             - 핵심 정보만 5줄 내외로 간결하게 정리하세요

            검색 결과 요약:
            """
        )

        # 최종 답변용 프롬프트
        self.answer_prompt = ChatPromptTemplate.from_template(
            """
            아래에 어떤 질문에 대한 웹 검색을 수행한 뒤 정리한 요약 내용입니다

            [검색 요약]
            {mysummary}

            [질문]
            {myquestion}

            위 요약에 있는 정보만 사용해서 질문에 답변하세요.
            추측하거나 지어내지 말고, 모르면 모른다고 말하세요.

            최종 답변:
            """
        )

    def summarize_search(self, question):   # Tavily로 실시간 웹 검색
        raw_results = self.search.invoke(
            {"query":question}
        )

        time.sleep(0.5)
        chain = self.summary_prompt | self.llm

        # LLM에게 요약 생성
        summary_msg = chain.invoke({"search_results":raw_results})
        return summary_msg.content

    def answerFunc(self, question):
        print(f'검색 및 요약 중 : {question}')

        # 1단계 : 질문에 대해 웹 검색 후 요약 생성
        summary = self.summarize_search(question)

        # 2단계 : 검색 결과과 요약된 내용을 바탕으로 최종 답변 생성
        chain = self.answer_prompt | self.llm

        answer_msg = chain.invoke(
            {
                "mysummary":summary,
                "myquestion":question
            }
        )
        return answer_msg.content

if __name__ == '__main__':
    web_rag = MyOptimizedWebRag()

    questions = [
        "최신 AI 기술은 뭐니?",
        "한국에서 가장 인기 있는 짬뽕은?",
        "한국의 여름철 장마 기간 중 점심때, 강남 근처에서 먹기 좋은 음식을 추천해"
    ]

    for q in questions:
        print(f'\n질문:{q}')
        answer = web_rag.answerFunc(q)
        print(f'답변:{answer}')
        print()
